In [1]:
!pip install -U transformers accelerate bitsandbytes sentencepiece tavily-python trafilatura

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uninstalling transformers-4.57.3:
      Successfully uninstalled transformers-4.57.3


In [36]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
import logging
logging.getLogger("trafilatura").setLevel(logging.CRITICAL)

import pandas as pd
import torch
import json
import re
import os
from typing import Annotated, TypedDict, Union, List
import operator
from google.colab import userdata

from tqdm import tqdm
import random
from sklearn.metrics import accuracy_score, f1_score

from tavily import TavilyClient
import trafilatura
import time


RANDOM_SEED = 42

from transformers import AutoTokenizer, AutoModelForCausalLM

from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, END

In [3]:
print("Загрузка модели Qwen/Qwen2.5-7B-Instruct...")

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True
)
print("Модель успешно загружена!")


Загрузка модели Qwen/Qwen2.5-7B-Instruct...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Модель успешно загружена!


In [5]:
data_path = '/content/drive/MyDrive/DLS_II_(NLP)/20_project_llm_agent_ya_search/data_final_for_dls_new.jsonl'
data_df = pd.read_json(data_path, lines=True)
train_df = data_df[570:]
eval_df = data_df[:570]
eval_df = eval_df[eval_df["relevance"] != 0.1]

In [6]:
def prep_df(df):
    initial_len = len(df)

    # Удаляем дубликаты
    df = df.drop_duplicates(subset=['Text', 'name', 'address'])

    ambiguous_count = len(df[~df['relevance'].isin([0.0, 1.0])])
    df = df[df['relevance'].isin([0.0, 1.0])]
    if ambiguous_count > 0:
        print(f"Удалено {ambiguous_count} строк с нечеткой релевантностью (например, 0.1)")

    if len(df) < initial_len:
        print(f"Всего удалено (дубликаты + спорные): {initial_len - len(df)}")

    # Удаляем старый столбец relevance, если он существует
    if 'relevance' in df.columns:
        df = df.drop(columns=['relevance'])
        print("Удален старый столбец 'relevance'")

    # Переименовываем relevance_new в relevance, если он существует
    if 'relevance_new' in df.columns:
        df = df.rename(columns={'relevance_new': 'relevance'})
        print("Столбец 'relevance_new' переименован в 'relevance'")

    return df

In [7]:
train_df = prep_df(train_df)

Удалено 4633 строк с нечеткой релевантностью (например, 0.1)
Всего удалено (дубликаты + спорные): 4638
Удален старый столбец 'relevance'
Столбец 'relevance_new' переименован в 'relevance'


In [14]:
try:
    TAVILY_KEY = userdata.get('Tavily')
except:
    TAVILY_KEY = input("Введите Tavily API Key: ")

tavily_client = TavilyClient(api_key=TAVILY_KEY)

In [28]:
def local_llm_invoke(messages: List[BaseMessage]):
    """
    Превращает сообщения LangChain в формат Qwen и генерирует ответ.
    Возвращает объект, имитирующий AIMessage (с атрибутом .content).
    """
    # Конвертация сообщений
    formatted_messages = []
    for msg in messages:
        role = "user"
        if isinstance(msg, SystemMessage):
            role = "system"
        elif isinstance(msg, AIMessage):
            role = "assistant"

        formatted_messages.append({"role": role, "content": msg.content})

    # Токенизация
    text = tokenizer.apply_chat_template(
        formatted_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # Генерация
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512,
            temperature=0.1,
            do_sample=False
        )

    # 4. Декодинг (убираем промпт из ответа)
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Возвращаем объект, у которого есть .content (как у LangChain)
    return AIMessage(content=response_text)

In [34]:
def clean_query_logic(name, address):
    clean_name = name.split(';')[0].strip()
    city = address.split(',')[0].strip()
    if "Россия" in city and len(address.split(',')) > 1:
        city = address.split(',')[1].strip()
    return f"{clean_name} {city} официальный сайт услуги"

def tool_web_search(query: str, verbose=False):
    if verbose:
        print(f"(Tavily) Ищу: {query}...")
    try:
        results = tavily_client.search(query, search_depth="advanced", max_results=2)
        urls = [r['url'] for r in results['results']]

        combined_text = ""
        for url in urls:
            if "youtube" in url or "vk.com" in url: continue
            try:
                downloaded = trafilatura.fetch_url(url)
                if downloaded:
                    text = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
                    if text:
                        combined_text += f"\n--- SOURCE: {url} ---\n{text[:600]}...\n"
            except:
                pass

        return combined_text if combined_text else "Сайты найдены, но прочитать не удалось."
    except Exception as e:
        return f"Ошибка поиска: {e}"

In [38]:
class AgentState(TypedDict):
    query: str
    org_name: str
    org_address: str
    org_category: str
    web_context: str
    iterations: int
    decision: dict
    thoughts: List[str]

def node_analyst(state: AgentState):
    iteration = state.get("iterations", 0)
    web_info = state.get("web_context", "")

    system_prompt = f"""
Ты — AI-агент, проверяющий данные.
ЗАДАЧА: Проверь, соответствует ли организация запросу.

ПРИМЕРЫ:
ПРИМЕР 1:

ЗАПРОС: "недорогие кафе геленджика"
ОРГАНИЗАЦИЯ: "Весёлая Кума"
КАТЕГОРИЯ: "Кафе"

НАЙДЕНО В ИНТЕРНЕТЕ:
"Уютное кафе на берегу моря с демократичными ценами."

ОТВЕТ:
{
  "reasoning": "Организация является кафе, в описании указаны демократичные цены, что соответствует запросу.",
  "action": "finish",
  "search_query": null,
  "verdict": 1
}
ПРИМЕР 2:

ЗАПРОС: "частная массажистка бутово"
ОРГАНИЗАЦИЯ: "Медицинский центр Здоровье"
КАТЕГОРИЯ: "Медцентр"

НАЙДЕНО В ИНТЕРНЕТЕ:
"Медицинские услуги, массаж, физиотерапия."

ОТВЕТ:
{
  "reasoning": "Запрос требует частную массажистку, а не медицинский центр. Прямого соответствия нет.",
  "action": "finish",
  "search_query": null,
  "verdict": 0
}

ТЕКУЩАЯ ЗАДАЧА:
ЗАПРОС: "{state['query']}"
ОРГАНИЗАЦИЯ: "{state['org_name']}"
АДРЕС: {state['org_address']}
КАТЕГОРИЯ: {state['org_category']}

НАЙДЕНО В ИНТЕРНЕТЕ:
{web_info if web_info else "Нет информации."}

ИНСТРУКЦИЯ:
1. Если информации мало -> action: "search".
2. Если уверен -> action: "finish".
3. Отвечай СТРОГО JSON. Без лишнего текста.

ПРИМЕР ОТВЕТА JSON:
{{
    "reasoning": "Я нашел сайт, там написано...",
    "action": "finish",
    "search_query": null,
    "verdict": 1
}}
ИЛИ
{{
    "reasoning": "Нужно проверить услуги...",
    "action": "search",
    "search_query": "{state['org_name']} услуги",
    "verdict": 0
}}
"""
    # Hard limit
    if iteration >= 2:
        system_prompt += "\nЛИМИТ ПОИСКА ИСЧЕРПАН. СРОЧНО ЗАВЕРШАЙ. action: finish"

    messages = [SystemMessage(content=system_prompt)]

    response = local_llm_invoke(messages)

    try:
        content = response.content.replace("```json", "").replace("```", "").strip()
        decision = json.loads(content)
        thoughts = state.get("thoughts", [])
        thoughts.append(decision.get("reasoning", ""))

    except:
        print(f"Ошибка парсинга JSON. Ответ модели: {response.content}")
        decision = {"action": "finish", "verdict": 0, "reasoning": "JSON Parse Error"}

    return {
        "messages": [response],
        "decision": decision,
        "thoughts": thoughts,
        "iterations": iteration + 1,
        "web_context": state.get("web_context", "")
    }

def node_searcher(state: AgentState):
    decision = state.get("decision", {})
    query = decision.get("search_query")

    if not query:
        query = clean_query_logic(state['org_name'], state['org_address'])

    new_info = tool_web_search(query)

    updated_context = f"""
=== ПОИСК ===
Запрос: {query}
Рез-т:
{new_info[:1500]}
"""

    return {
        "web_context": updated_context
    }

def router(state: AgentState):
    # HARD STOP
    if state["iterations"] >= 3:
        return "end"
    decision = state.get("decision", {})
    if decision.get("action") == "search":
        return "search"
    return "end"


Сборка агента

In [30]:
workflow = StateGraph(AgentState)
workflow.add_node("analyst", node_analyst)
workflow.add_node("search_tool", node_searcher)
workflow.set_entry_point("analyst")

workflow.add_conditional_edges(
    "analyst",
    router,
    {"search": "search_tool", "end": END}
)
workflow.add_edge("search_tool", "analyst")
app = workflow.compile()

Пробный прогон агента с полным выводом его узлов.

In [31]:
y_true = []
y_pred = []

for idx, row in eval_df.iterrows():
    print(f"\n🔹 {row['name']} | Запрос: {row['Text']}")

    final_state = None

    for event in app.stream({
        "query": row['Text'],
        "org_name": row['name'],
        "org_address": row['address'],
        "org_category": row['normalized_main_rubric_name_ru'],
        "web_context": "",
        "iterations": 0,
        "thoughts": [],
        "decision": {}
    }):
        for node_name, state in event.items():
            print(f"\nNODE: {node_name}")
            print("iterations:", state.get("iterations"))
            print("decision:", state.get("decision"))
            print("thoughts:", state.get("thoughts"))
            if state.get("web_context"):
                print("web_context:", state["web_context"][:300])
            else:
                print("web_context: <empty>")

            final_state = state

    gt = int(row["relevance"])
    pred = int(final_state["decision"].get("verdict", 0))

    y_true.append(gt)
    y_pred.append(pred)

    print("\n FINAL RESULT")
    print(f"PREDICTED: {pred}")
    print(f"GROUND TRUTH: {gt}")
    print("ВЕРНО" if pred == gt else "НЕВЕРНО")

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("\n" + "="*50)
print("FINAL METRICS")
print(f"Accuracy: {acc:.3f}")
print(f"F1-score: {f1:.3f}")
print("="*50)


🔹 Весёлая Кума; Vesolaya kuma; Весёлая кума; Ресторан Весёлая Кума | Запрос: недорогие кафе геленджика

NODE: analyst
iterations: 1
decision: {'reasoning': 'Нет информации о стоимости услуг организации в найденных данных.', 'action': 'search', 'search_query': 'Весёлая Кума; Vesolaya kuma; Весёлая кума; Ресторан Весёлая Кума цены', 'verdict': 0}
thoughts: ['Нет информации о стоимости услуг организации в найденных данных.']
web_context: <empty>
(Tavily) Ищу: Весёлая Кума; Vesolaya kuma; Весёлая кума; Ресторан Весёлая Кума цены...

NODE: search_tool
iterations: None
decision: None
thoughts: None
web_context: 
=== ПОИСК ===
Запрос: Весёлая Кума; Vesolaya kuma; Весёлая кума; Ресторан Весёлая Кума цены
Рез-т:

--- SOURCE: https://gelenjik.ugosti.com/vesyolaya-kuma ---
Уютное кафе на берегу моря с прекрасным душевным интерьером и замечательной домашней кухней. Очень демократичные цены и отличное непринуждён

NODE: analyst
iterations: 2
decision: {'reasoning': 'На сайте указано, что это уютно


NODE: search_tool
iterations: None
decision: None
thoughts: None
web_context: 
=== ПОИСК ===
Запрос: Трансмост-тур; Transmost-tour; Transmost-tur железнодорожные билеты
Рез-т:

--- SOURCE: https://transmost-tour.ru/ ---
╬ёэютэющ тшф фх Ґхы№эюёҐш ъюьярэшш л╥ЁрэёьюёҐ-ҐґЁ╗ Ц яЁхфюёҐртыхэшх ёютЁхьхээ√є ш ъріхёҐтхээ√є ґёыґу яю яЁюфрцх ш фюёҐртъх ртшрсшыхҐют ш ╞/─ сшыхҐют, р Ґръцх 

NODE: analyst
iterations: 2
decision: {'reasoning': 'Найденный текст на сайте не содержит прямых упоминаний о железнодорожных билетах, хотя это указано в категории запроса. Необходимо проверить дополнительную информацию.', 'action': 'search', 'search_query': 'Трансмост-тур; Transmost-tour; Transmost-tur железнодорожные билеты', 'verdict': 0}
thoughts: ['Нет информации о предоставлении железнодорожных билетов.', 'Найденный текст на сайте не содержит прямых упоминаний о железнодорожных билетах, хотя это указано в категории запроса. Необходимо проверить дополнительную информацию.']
web_context: 
=== ПОИСК ===
Запро


NODE: search_tool
iterations: None
decision: None
thoughts: None
web_context: 
=== ПОИСК ===
Запрос: Трансмост-тур; Transmost-tour; Transmost-tur железнодорожные билеты
Рез-т:

--- SOURCE: https://transmost-tour.ru/ ---
╬ёэютэющ тшф фх Ґхы№эюёҐш ъюьярэшш л╥ЁрэёьюёҐ-ҐґЁ╗ Ц яЁхфюёҐртыхэшх ёютЁхьхээ√є ш ъріхёҐтхээ√є ґёыґу яю яЁюфрцх ш фюёҐртъх ртшрсшыхҐют ш ╞/─ сшыхҐют, р Ґръцх 

NODE: analyst
iterations: 3
decision: {'reasoning': 'На сайте указано, что организация занимается продажей железнодорожных билетов и предоставляет другие услуги, включая железнодорожные билеты из Казахстана в Москву.', 'action': 'finish', 'verdict': 1}
thoughts: ['Нет информации о предоставлении железнодорожных билетов.', 'Найденный текст на сайте не содержит прямых упоминаний о железнодорожных билетах, хотя это указано в категории запроса. Необходимо проверить дополнительную информацию.', 'На сайте указано, что организация занимается продажей железнодорожных билетов и предоставляет другие услуги, включая железн


NODE: search_tool
iterations: None
decision: None
thoughts: None
web_context: 
=== ПОИСК ===
Запрос: Milwaukee Москва Павловская улица 27с4
Рез-т:
Сайты найдены, но прочитать не удалось.


NODE: analyst
iterations: 2
decision: {'reasoning': 'Нет информации о наличии слесарного разметочного инструмента в указанном адресе.', 'action': 'search', 'search_query': 'Milwaukee слесарный разметочный инструмент Москва Павловская улица 27с4', 'verdict': 0}
thoughts: ['Нет информации о организации Milwaukee в Москве, Павловская улица, 27с4.', 'Нет информации о наличии слесарного разметочного инструмента в указанном адресе.']
web_context: 
=== ПОИСК ===
Запрос: Milwaukee Москва Павловская улица 27с4
Рез-т:
Сайты найдены, но прочитать не удалось.

(Tavily) Ищу: Milwaukee слесарный разметочный инструмент Москва Павловская улица 27с4...

NODE: search_tool
iterations: None
decision: None
thoughts: None
web_context: 
=== ПОИСК ===
Запрос: Milwaukee слесарный разметочный инструмент Москва Павловская ули

NameError: name 'f1_score' is not defined

Прогон eval_df.

In [39]:
y_true = []
y_pred = []

print("\nEVALUATION RESULTS")
print("-" * 60)

for idx, row in eval_df.iterrows():
    res = app.invoke({
        "query": row['Text'],
        "org_name": row['name'],
        "org_address": row['address'],
        "org_category": row['normalized_main_rubric_name_ru'],
        "web_context": "",
        "iterations": 0,
        "thoughts": [],
        "decision": {}
    })

    gt = int(row["relevance"])
    pred = int(res["decision"].get("verdict", 0))

    y_true.append(gt)
    y_pred.append(pred)

    print(
        f"Запрос: {row['Text']} | "
        f"Организация: {row['name']} | "
        f"Pred: {pred} | "
        f"GT: {gt}"
    )

acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("-" * 60)
print(f"Accuracy: {acc:.3f}")
print(f"F1-score: {f1:.3f}")
print("-" * 60)


EVALUATION RESULTS
------------------------------------------------------------


KeyboardInterrupt: 